# 80 km/h, retour aux 90 km/h : quel effet sur la mortalité routière ?

**Projet Python pour la data science — ENSAE 2A, 2026-2027**
Héloïse Gonon, Samir …, Brice … *(à compléter)*

> **À rédiger en dernier : résumé en 5 lignes** (question, données, méthode, résultat principal, principale limite).

## Comment exécuter ce notebook

1. Installer les dépendances : `pip install -r requirements.txt`.
2. Lancer toutes les cellules dans l'ordre (« Run All »).

Au premier lancement, les données sont téléchargées depuis leurs sources publiques (data.gouv.fr, INSEE, SDES, cartiflette) puis nettoyées : compter **XX minutes** *(à mesurer)*. Les fichiers produits sont gardés dans `data/processed/` (non versionné) et relus aux lancements suivants. Le code est dans `scripts/` ; ce notebook ne fait qu'appeler les fonctions et commenter les résultats.

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

sys.path.insert(0, str(Path("scripts").resolve()))

import data_analysis as da
import graphiques as gr
import indicateurs as ind
from collecte import charger_contours, charger_departements_90, charger_population, charger_radars

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
gr.appliquer_style()

# Variable principale pour Q2 : tués sur les routes départementales hors agglomération
VARIABLE_Q2 = "tues_rd"

## 1. Introduction

### 1.1 Contexte

> **À rédiger** (reprendre la section 2 de `docs/plan_projet.md` en 10-15 lignes) : passage à 80 km/h le 1er juillet 2018, possibilité de revenir aux 90 km/h depuis la LOM (décembre 2019), environ la moitié des départements l'ont fait, à des dates et avec une ampleur différentes. D'où une **expérience naturelle**.

### 1.2 Question et hypothèses

**La baisse de la vitesse autorisée à 80 km/h a-t-elle réduit la mortalité routière, et le retour aux 90 km/h l'a-t-il fait remonter ?**

| Hypothèse | Ce qu'on devrait observer si elle est vraie |
|---|---|
| H1 : le 80 km/h a réduit la mortalité | Rupture à la baisse en 2018-2019 sur le réseau concerné, absente ou plus faible ailleurs |
| H2 : le retour aux 90 km/h a augmenté la mortalité | Après relèvement, évolution moins favorable dans les départements relevés que dans ceux restés à 80 |
| H3 : l'effet dépend de l'intensité | Écart plus grand pour les relèvements totaux que partiels |
| H4 : l'effet passe par certains types d'accidents | Variation concentrée sur les chocs frontaux et les sorties de route |

> ⚠️ Nous mesurons l'effet d'un **changement de règle**, pas d'un changement de vitesse observée : les vitesses pratiquées ne sont pas disponibles.

## 2. Données

### 2.1 Sources

| Source | Producteur | Accès dans le code | Rôle |
|---|---|---|---|
| Accidents corporels (BAAC) 2010-2024 | ONISR | API data.gouv.fr (`collecte.telecharger_baac`) | Tués, accidents |
| Départements revenus aux 90 km/h | **Construite par nous** | `data/departements_90.csv` (versionné) | Groupes traités / témoins |
| Population par département | INSEE | API Melodi (`charger_population`) | Taux par habitant |
| Ventes de carburants routiers | SDES | API DiDo (`charger_carburants`) | Indicateur du trafic (2020) |
| Radars fixes | Ministère de l'Intérieur | data.gouv.fr (`charger_radars`) | Q3 (annexe) |
| Contours des départements | IGN | `cartiflette` | Cartes |

In [ ]:
accidents, panel = ind.charger_ou_construire()
groupes = ind.affecter_groupes(charger_departements_90())
population = charger_population()
contours = charger_contours()

print(f"{len(accidents):,} accidents de {accidents['annee'].min()} à {accidents['annee'].max()}")
print(f"Panel : {panel['code_dep'].nunique()} départements × {panel['date_mois'].nunique()} mois")

### 2.2 La table des départements revenus aux 90 km/h

Aucune table officielle ne recense proprement les relèvements : nous l'avons construite à partir de l'étude ONISR de février 2024 (base officielle), de la liste L'Argus 2026, de la liste de la Ligue de défense des conducteurs et de la presse locale. La méthode complète et les points encore incertains sont dans `docs/collecte_departements.md`.

**Groupes utilisés.** `G90_total` : au moins 80 % du réseau départemental relevé ; `G90_partiel` : relèvement partiel ; `G80` : aucun relèvement avant fin 2024 (Morbihan et Eure, relevés en 2025-2026, sont donc dans `G80`).

In [ ]:
display(groupes["groupe"].value_counts().rename("départements").to_frame())
display(
    groupes.loc[groupes["groupe"] != "G80", ["nom_dep", "groupe", "date_releve_eff", "precision_date", "part_releve"]]
    .sort_values("date_releve_eff")
    .head(10)
)

> **À rédiger.** Combien de départements dans chaque groupe ? Quand ont eu lieu la plupart des relèvements ? Quelles incertitudes restent sur la table (dates approximatives, annulations en justice, écarts de linéaire entre sources) et quel effet peuvent-elles avoir sur nos résultats ?

### 2.3 Nettoyage et harmonisation des BAAC

Traitements codés dans `scripts/clean_data.py` :

- **séparateur et encodage** détectés pour chaque fichier (ils changent selon les années) ;
- **codes département** des anciens fichiers ramenés au format officiel (`590` → `59`, `201` → `2A`) ;
- **coordonnées** d'avant 2019 converties (entiers divisés par 100 000) et valeurs aberrantes retirées ;
- **valeurs « non renseigné »** (`-1`) converties en valeurs manquantes ;
- usagers et véhicules **agrégés au niveau de l'accident**, puis fusion des quatre tables.

**Réseau concerné par le 80 km/h** : routes nationales ou départementales, hors agglomération, à double sens. Pour Q2, on se restreint aux **routes départementales** (`catr == 3`), les seules que les départements peuvent relever. Indicateur principal : les **tués à 30 jours** (les blessés hospitalisés ne sont plus fiables depuis 2018).

> **À compléter** : les usagers en fuite (ajoutés aux BAAC en 2021) ne sont pas retirés ; dire si c'est un problème pour les tués et pourquoi.

In [ ]:
controle = ind.tableau_controle_annuel(accidents)
display(controle.round(3))
gr.tracer_description_annuelle(controle);

> **À rédiger.** Y a-t-il un saut artificiel en 2019, l'année du changement de format ? La part d'accidents sur le réseau concerné est-elle stable ? Depuis 2019, quelle part des accidents classés « réseau concerné » a bien une VMA de 80 ou 90 km/h, et qu'est-ce que cela dit de notre règle ?

In [ ]:
gr.heatmap_manquants(ind.valeurs_manquantes(accidents));

> **À rédiger.** Quelles variables sont souvent manquantes, et à partir de quand ? (`vma` n'existe qu'à partir de 2019 : c'est attendu.) Les variables de la règle du réseau (`catr`, `agg`, `circ`) sont-elles assez complètes ?

In [ ]:
comparaison = ind.verifier_coherence_onisr(accidents, charger_departements_90())
display(comparaison[["tues_2019_calc", "tues_ha_2019_onisr", "tues_2022_calc", "tues_ha_2022_onisr"]].sum().to_frame("total"))

> **À rédiger.** Nos tués hors agglomération hors autoroute retrouvent-ils les chiffres de l'ONISR ? Si un écart existe, d'où peut-il venir ?

## 3. Le passage à 80 km/h en 2018 (Q1, H1)

In [ ]:
gr.tracer_mensuel_reseau(ind.tues_mensuels_par_reseau(accidents));

In [ ]:
bilan = da.bilan_national(panel)
display(bilan[["tues_80", "tues_temoin", "indice_tues_80", "indice_tues_temoin", "indice_carburants_routiers_m3"]].round(1))

> **À rédiger.** La mortalité baisse-t-elle davantage après juillet 2018 sur le réseau concerné que sur les autres réseaux ? La baisse de 2020 suit-elle la baisse du trafic (carburants) ? Conclure sur **H1**, avec prudence : d'autres facteurs ont pu jouer en même temps.

## 4. Le retour aux 90 km/h (Q2)

### 4.1 Qui a relevé, et quand ?

In [ ]:
gr.carte_groupes(contours, groupes[["code_dep", "groupe"]]);

In [ ]:
gr.tracer_calendrier(groupes.assign(date_releve=groupes["date_releve_eff"]));

> **À rédiger.** Les départements relevés sont-ils géographiquement concentrés (centre, rural, montagne) ? Que cela implique-t-il pour la comparabilité avec les départements restés à 80 (biais de sélection) ? Les relèvements sont-ils simultanés ou échelonnés ?

### 4.2 Évolution comparée des groupes (graphique central)

Tués sur les routes départementales hors agglomération, par million d'habitants, en indice (moyenne 2015-2017 = 100). La bande colorée est un intervalle à 95 % (approximation de Poisson).

In [ ]:
serie_g4 = ind.serie_groupes_ic(panel, variable=VARIABLE_Q2, par_habitant=True)
gr.tracer_series_groupes_ic(
    serie_g4,
    titre="G4. Tués sur les routes départementales hors agglomération, par groupe",
    ylabel="Indice des tués par habitant (2015-2017 = 100)",
);

> **À rédiger.** **Avant 2020**, les courbes sont-elles parallèles (condition de validité de la comparaison) ? **Après 2020**, les départements relevés s'écartent-ils de ceux restés à 80, et dans quel sens ? L'écart est-il plus fort pour le relèvement total (**H3**) ? Les intervalles se chevauchent-ils ?

### 4.3 Étude d'événement : chaque département aligné sur sa propre date

Pour chaque département relevé, l'année 0 est celle de son relèvement. On compare ses tués à ce qu'il aurait connu s'il avait suivi la même évolution que les départements restés à 80 km/h (base : moyenne des années −3 à −1). 100 = même évolution que les G80 ; au-dessus de 100 = évolution moins favorable.

In [ ]:
evenement = ind.etude_evenement(panel, variable=VARIABLE_Q2)
gr.tracer_evenement(evenement);
display(evenement.pivot(index="annee_relative", columns="groupe", values="nb_departements"))

> **À rédiger.** L'indice est-il stable autour de 100 avant le relèvement ? Monte-t-il après ? Attention : le nombre de départements diminue pour les dernières années (tableau ci-dessus), ces points sont donc plus fragiles.

### 4.4 Par type d'accident (H4)

In [ ]:
categories = ind.evolution_par_categorie(accidents, groupes, categorie="type_collision")
gr.tracer_evolution_categories(
    categories, "type_collision", ind.LIBELLES_COLLISION,
    titre="G6. Évolution des tués par type de collision (routes départementales hors agglomération)",
);

> **À rédiger.** Les évolutions diffèrent-elles entre groupes surtout pour les collisions frontales et les accidents sans collision (sorties de route) ? Conclure sur **H4**. Rappeler que certains départements relevés en 2022-2023 sont comptés « après » dès 2021.

### 4.5 Cartes avant / après

In [ ]:
taux = ind.taux_avant_apres_par_dep(accidents, population, perimetre="rd")
gr.cartes_avant_apres(contours, taux);

> **À rédiger.** Où la mortalité par habitant est-elle la plus forte ? Les hausses se concentrent-elles dans les départements relevés (comparer avec la carte des groupes) ? Attention aux petits départements, où quelques décès changent beaucoup le taux.

## 5. Démarche : différence de différences et robustesse

Pour chaque département relevé, on compare les tués sur une fenêtre avant et une fenêtre après son relèvement (même longueur, multiple de 12 mois pour garder la même saisonnalité), et on retire l'évolution des départements restés à 80 km/h **sur les mêmes mois** :

$$\text{effet} = \underbrace{(\text{évolution des départements relevés})}_{\%} - \underbrace{(\text{évolution des G80 sur les mêmes mois})}_{\%}$$

Les mois de confinement strict (mars-mai et novembre 2020) sont exclus.

In [ ]:
avant_apres = da.avant_apres_releve(panel, fenetre=24, variable=VARIABLE_Q2)
display(da.synthese_avant_apres(avant_apres))

> **À rédiger.** Lire le tableau : de combien les tués ont-ils évolué dans chaque groupe, et chez les G80 sur les mêmes mois ? Quel est l'écart ?

In [ ]:
robustesse = ind.tableau_robustesse(panel, variable=VARIABLE_Q2)
display(robustesse.round(1))
gr.tracer_robustesse(robustesse);

> **À rédiger.** Le signe de l'effet est-il le même dans toutes les configurations ? Les intervalles (`ic_bas`, `ic_haut`) contiennent-ils 0 ? Une conclusion qui tient partout est solide ; une conclusion qui ne tient que dans une configuration doit être présentée avec prudence.

### 5.1 Comparaison avec l'évaluation de l'ONISR

> **À rédiger.** Résumer la méthode de l'ONISR (groupes, période 2019-2022, indicateur), comparer ses groupes et ses ordres de grandeur aux nôtres, expliquer les écarts (définition des groupes, période, traitement de 2020).

## 6. Q3 (optionnel) : les radars fixes

Pour chaque radar fixe mis en service entre 2020 et 2022, on compte les accidents à moins de 1 km pendant les 2 ans qui précèdent et les 2 ans qui suivent sa mise en service. Le nombre « attendu » applique aux accidents d'avant l'évolution nationale entre les deux mêmes fenêtres (Covid, tendance générale) : un radar est efficace s'il fait mieux que cette tendance.

Seuls les accidents géolocalisés sont utilisables ; avant 2015, les coordonnées manquent pour environ deux tiers d'entre eux, d'où le choix des radars récents.

In [ ]:
radars = charger_radars()
radars_proches = ind.accidents_autour_radars(accidents, radars)
gr.tracer_radars(radars_proches)
radars_proches.groupby("type_radar")[["avant", "apres", "attendu_apres"]].sum().assign(
    radars=radars_proches["type_radar"].value_counts()
)

> **À rédiger.** ETT = radar tourelle, ETD = radar discriminant, ETF = radar fixe. Lire le graphique : l'évolution observée des accidents autour des radars est-elle meilleure que celle attendue d'après la tendance nationale ? Le résultat diffère-t-il selon le type de radar, et l'écart par radar est-il distinguable de zéro ?

À discuter avec prudence :

- **régression vers la moyenne** : les radars sont installés là où il y a eu beaucoup d'accidents, qui auraient de toute façon baissé ensuite ;
- **biais de survie** : le fichier ne contient que les radars encore en service ;
- un accident proche de deux radars est compté deux fois ;
- les tourelles (ETT) ne contiennent pas toujours un radar actif ;
- quelques accidents par radar seulement : les écarts individuels sont dominés par le bruit.

## 7. Conclusion

> **À rédiger.** Une phrase de conclusion par hypothèse (H1 à H4), y compris quand les données ne permettent pas de trancher.

**Limites :**

- nous mesurons l'effet d'une règle, pas de la vitesse réellement pratiquée ;
- les départements ont choisi de relever ou non : la comparaison n'est pas expérimentale et ne prouve pas la causalité, elle la rend plus ou moins plausible ;
- 2020 (Covid) coïncide avec les premiers relèvements ;
- les nombres de tués par département sont petits ; les dates de certains relèvements sont incertaines ;
- les résultats valent pour le réseau secondaire français entre 2015 et 2024.